In [1]:
# Licensed under a 3-clause BSD style license - see LICENSE.rst
"""Spectrum 1D ON/OFF Analysis"""

'Spectrum 1D ON/OFF Analysis'

# Spectrum 1D ON/OFF Analysis with 1LHAASO Catalog


This notebook demonstrates how to:
1. Import necessary libraries and modules.
2. Load and select a source from the 1LHAASO catalog.
3. Configure an `AnalysisSpectrumConfig` object.
4. Perform spectrum analysis using `AnalysisSpectrum`.
5. Save and inspect the results, including flux points and fit results.

---


In [2]:
# Standard library imports
import astropy.units as u

# Third-party imports from gammapy
from gammapy.catalog import SourceCatalog1LHAASO
from gammapy.data import Observation
from gammapy.datasets import Datasets
from gammapy.utils.scripts import make_path
from gammapy.modeling.models import PowerLawSpectralModel, SkyModel, Models

# Third-party imports from regions
from regions import CircleSkyRegion

# feupy module imports
from feupy.utils.coordinates import convert_skycoord_to_dict
from feupy.analysis.config import CTAOAnalysisConfig
from feupy.analysis.core import CTAOAnalysis
from feupy.visualization.counts import show_hist_counts


### 1. Load and Select Source
In this section, we load the 1LHAASO catalog and select a specific source (1LHAASO J1219+2915) for analysis.

In [3]:
# Load the 1LHAASO catalog and select a source
catalog = SourceCatalog1LHAASO()
catalog.table

Source_Name,Model_a,RAJ2000,DECJ2000,pos_err,r39,r39_err,r39_ul,N0,N0_err,N0_ul,gamma,gamma_err,gamma_ul,E0,TS,TS100,Model_b,RAJ2000_b,DECJ2000_b,pos_err_b,r39_b,r39_err_b,r39_ul_b,N0_b,N0_err_b,N0_ul_b,gamma_b,gamma_err_b,gamma_ul_b,E0_b,TS_b,TS100_b,ASSO_Name,ASSO_Sep
,,deg,deg,deg,deg,deg,deg,1 / (TeV s cm2),1 / (TeV s cm2),1 / (TeV s cm2),,,,TeV,,,,deg,deg,deg,deg,deg,deg,1 / (TeV s cm2),1 / (TeV s cm2),1 / (TeV s cm2),,,,TeV,,,,deg
bytes20,bytes7,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,bytes7,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,bytes16,float64
1LHAASO J0007+5659u,KM2A,1.86,57.0,0.12,--,--,0.18,3.3e-17,5e-18,--,3.1,0.2,--,50.0,86.5,43.6,WCDA,--,--,--,--,--,--,--,--,2.7000000000000002e-14,--,--,--,3.0,--,--,--,--
1LHAASO J0007+7303u,KM2A,1.91,73.07,0.07,0.17,0.03,--,3.41e-16,2.7e-17,--,3.4,0.12,--,50.0,361.0,171.6,WCDA,1.48,73.15,0.1,--,--,0.22,5.01e-13,1.1100000000000002e-13,--,2.74,0.11,--,3.0,141.6,--,CTA 1,0.12
1LHAASO J0056+6346u,KM2A,14.1,63.77,0.08,0.24,0.03,--,1.47e-16,1e-17,--,3.33,0.1,--,50.0,380.2,94.1,WCDA,13.78,63.96,0.15,0.33,0.07,--,1.45e-13,4.0999999999999996e-14,--,2.35,0.13,--,3.0,106.1,--,--,--
1LHAASO J0206+4302u,KM2A,31.7,43.05,0.13,--,--,0.27,2.4e-17,3e-18,--,2.62,0.16,--,50.0,96.0,82.8,WCDA,--,--,--,--,--,--,--,--,9e-15,--,--,--,3.0,--,--,--,--
1LHAASO J0212+4254u,KM2A,33.01,42.91,0.2,--,--,0.31,1.2e-17,3e-18,--,2.45,0.23,--,50.0,38.4,30.2,WCDA,--,--,--,--,--,--,--,--,7.000000000000001e-15,--,--,--,3.0,--,--,--,--
1LHAASO J0216+4237u,KM2A,34.1,42.63,0.1,--,--,0.13,1.8e-17,3e-18,--,2.58,0.17,--,50.0,102.0,65.6,WCDA,--,--,--,--,--,--,--,--,2.0000000000000003e-14,--,--,--,3.0,--,--,--,--
1LHAASO J0249+6022,KM2A,42.39,60.37,0.16,0.38,0.08,--,9.300000000000001e-17,9e-18,--,3.82,0.18,--,50.0,148.8,--,WCDA,41.52,60.49,0.4,0.71,0.1,--,1.96e-13,5.1000000000000004e-14,--,2.52,0.16,--,3.0,53.3,--,--,--
1LHAASO J0339+5307,KM2A,54.79,53.13,0.11,--,--,0.22,5.799999999999999e-17,6e-18,--,3.64,0.16,--,50.0,144.0,--,WCDA,--,--,--,--,--,--,--,--,2.1e-14,--,--,--,3.0,--,--,LHAASOJ0341+5258,0.37


In [4]:
source = catalog["1LHAASO J1219+2915"]

In [5]:
source.position

<SkyCoord (FK5: equinox=J2000.000): (ra, dec) in deg
    (184.98, 29.25)>

In [6]:
source.name

'1LHAASO J1219+2915'

In [7]:
source.sky_model()

SkyModel(spatial_model=<gammapy.modeling.models.spatial.PointSpatialModel object at 0x7d4c50467290>, spectral_model=<gammapy.modeling.models.spectral.PowerLawSpectralModel object at 0x7d4c48bcf350>)temporal_model=None)

In [8]:
source.spectral_model("KM2A")

In [9]:
source.spectral_model("WCDA")

In [ ]:
from feupy.analysis.irfs import get_visibility_table_from_position

### Annual Visibility

In [ ]:
year=2025

get_visibility_table_from_position(source.position, year)

In [ ]:
source.position

### 2. Configure `CTAOAnalysis`
Here, we configure the `CTAOAnalysisConfig` object with observation, dataset, and analysis settings for the spectrum analysis.

In [10]:
# Create and configure CTAOAnalysisConfig
config = CTAOAnalysisConfig()
# Define observation settings
position = source.position.icrs

config.observation.obs_cone = convert_skycoord_to_dict(position)
# Define o "ponto central" (descontando o offset) da observação no céu, convertido para dicionário (RA, Dec).

config.observation.position_angle = 0 * u.deg 
# Ângulo de rotação da câmera em torno do centro do campo de visão. Aqui está fixo em 0°.
config.observation.offset = 0.5 * u.deg 
# Offset radial entre o centro da observação e o centro da região ON. Simula observações em “wobble mode”.
config.observation.livetime = 50 * u.h 
# Tempo de exposição total (duração da observação simulada).
config.observation.required_irfs = ["South", "AverageAz", "40deg", "50h"]
# Define qual conjunto de IRFs será usada:

# South: observatório do hemisfério sul (CTAO-Sul)
# AverageAz: IRFs médios em azimute
# 40deg: zenital
# 50h: IRFs optimizadas para 50 horas de observação

# Configure dataset settings
config.datasets.map_selection = ["edisp", "background", "exposure"]
# Significado:
# Essa linha define quais componentes do instrumento devem ser usados na criação do dataset. 
# Cada item é uma "camada" que pode ser incluída no modelo da simulação ou análise.

#     "edisp" → Energy dispersion: aplica a matriz de dispersão de energia, que simula o desvio entre energia verdadeira e reconstruída.

#     "background" → adiciona o modelo de fundo (por exemplo, de ruído cósmico ou instrumental).

#     "exposure" → considera a exposição efetiva (produto da área efetiva e tempo de observação).

config.datasets.safe_mask.methods = ["aeff-default", "aeff-max"]
# Significado:
# Define o método usado para calcular a região segura de energia (safe_mask) no espectro.

#     "bkg-peak" significa que o Gammapy vai buscar o pico da contagem de fundo e considerar como confiável 
# somente a faixa de energia abaixo desse pico (onde o ruído não domina).
# Isso evita usar faixas de energia onde o modelo do fundo é muito incerto.

config.datasets.safe_mask.parameters = {"aeff_percent": 10}

# Significado:
# Esse parâmetro é complementar ao método de máscara segura e é usado, por exemplo, no método "aeff-default" ou "aeff-max" 
# (caso fosse escolhido). Ele define que a energia mínima confiável será aquela onde a área efetiva (effective area) 
# atinge 10% do valor máximo.
    
config.datasets.containment_correction = False
# Significado:
# Indica se o Gammapy deve aplicar uma correção de contenção angular para a região espectral.

#     True → corrige o fluxo para levar em conta que a PSF (função de espalhamento do ponto) pode jogar fótons fora da região analisada.

#     False → não aplica essa correção, assumindo que o modelo já considera a fração de fótons dentro da região.

# Recomendado deixar como False quando se usa espectros extraídos de regiões ON/OFF fixas e pequenas, como em CTA.

config.datasets.use_region_center = True
# Por padrão (True), o maker avalia os IRFs apenas no centro da região ON.
# Isto é válido apenas para fontes pontuais ou muito pequenas.
# Mas para fontes estendidas, isso ignoraria variações no IRF dentro da região.
# Com False, Gammapy faz o average dos IRFs por toda a região ON


config.datasets.on_region = convert_skycoord_to_dict(position)
config.datasets.on_region.radius = 0.5 * u.deg
# Define a região espectral ON: centro (posição da fonte) e raio (0.5°).

config.datasets.stack = False
# Indica que os datasets não serão empilhados. Útil para fazer análise por observação individual.

config.datasets.on_off.acceptance = 1
config.datasets.on_off.acceptance_off = 5
# Fatores de aceitação:

# ON = 1 → normalização da região ON

# OFF = 5 → a região OFF cobre 5× a área da ON (relacionado ao número de regiões OFF refletidas)


# Configure energy axes
config.datasets.geom.axes.energy.min = 30 * u.GeV
config.datasets.geom.axes.energy.max = 300 * u.TeV
config.datasets.geom.axes.energy.nbins = 12
# Define o range e número de bins da energia reconstruída, usada nos plots e ajuste.

config.datasets.geom.axes.energy_true.min = 3 * u.GeV
config.datasets.geom.axes.energy_true.max = 500 * u.TeV
config.datasets.geom.axes.energy_true.nbins = 15
# Faixa de energia usada internamente nas simulações (com dispersão de energia). Mais larga para capturar todos os eventos.


config.flux_points.energy.min = 30 * u.GeV
config.flux_points.energy.max = 300 * u.TeV
config.flux_points.energy.nbins = 12
config.flux_points.source = "source"
# Configura os pontos de fluxo (flux points) que podem ser calculados após o ajuste do espectro.
    
# Configure sensitivity settings
config.sensitivity.gamma_min = 5 # Número mínimo de fótons esperados (excesso) para considerar uma detecção.
config.sensitivity.n_sigma = 3 # Significância mínima (em sigma) para detecção.
config.sensitivity.bkg_syst_fraction = 0.10 # Erro sistemático do fundo (assumido como 10%) incluído no cálculo de sensibilidade.

config.statistics.n_obs = 1 # Número de observações

DATA_PATH = make_path(f'./data/')
DATA_PATH.mkdir(parents=True, exist_ok=True)

config.sensitivity.data_path = DATA_PATH
# Print configuration to verify
print(config)


CTAOAnalysisConfig

    general:
        log: {level: info, filename: null, filemode: null, format: null, datefmt: null}
        outdir: .
        n_jobs: 1
        datasets_file: null
        models_file: null
    observation:
        obs_cone: {frame: icrs, lon: 184.97999684579457 deg, lat: 29.250002038380092 deg,
            radius: null}
        livetime: 50.0 h
        offset: 0.5 deg
        position_angle: 0.0 deg
        required_irfs: [South, AverageAz, 40deg, 50h]
    datasets:
        type: 1d
        stack: false
        geom:
            wcs:
                skydir: {frame: null, lon: null, lat: null}
                binsize: 0.02 deg
                width: {width: 5.0 deg, height: 5.0 deg}
                binsize_irf: 0.2 deg
            selection: {offset_max: 2.5 deg}
            axes:
                energy: {min: 30.0 GeV, max: 300.0 TeV, nbins: 12}
                energy_true: {min: 3.0 GeV, max: 500.0 TeV, nbins: 15}
        map_selection: [edisp, background, exposu

### 3. Running Spectrum Analysis
In this section, we create an instance of `CTAOAnalysis` and perform the spectrum analysis based on the configured settings. The process includes simulating observations, running fits, and extracting flux points.

In [ ]:
# Licensed under a 3-clause BSD style license
"""Simplified CTAO high-level analysis (ON/OFF 1D, simulation-driven)."""

import logging
import numpy as np
import astropy.units as u

from astropy.coordinates import SkyCoord
from astropy.table import Table
from regions import CircleSkyRegion

from gammapy.datasets import (
    Datasets,
    SpectrumDataset,
    SpectrumDatasetOnOff,
    FluxPointsDataset,
)
from gammapy.maps import MapAxis, RegionGeom
from gammapy.makers import (
    SpectrumDatasetMaker,
    ReflectedRegionsBackgroundMaker,
    SafeMaskMaker,
    RingBackgroundMaker,
    FoVBackgroundMaker,
)
from gammapy.modeling import Fit
from gammapy.modeling.models import Models
from gammapy.estimators import FluxPointsEstimator, SensitivityEstimator
from feupy.utils.datasets import flux_points_dataset_from_table
from gammapy.estimators.points import FluxPoints

from gammapy.data import Observation, Observations
from gammapy.utils.scripts import make_path
from gammapy.data import FixedPointingInfo

from feupy.analysis.config import CTAOAnalysisConfig
from feupy.analysis.core import CTAOAnalysis
from feupy.irf.manager import CTAOIRFManager
from feupy.utils.table import write_tables_csv, write_tables_fits


In [11]:
# Create and run spectrum analysis
analysis = CTAOAnalysis(config)

# Simulate observation and spectrum
analysis.simulate_observation()

INFO:feupy.analysis.config:Setting logging config: {'level': 'INFO', 'filename': None, 'filemode': None, 'format': None, 'datefmt': None}
INFO:feupy.analysis.core:Observation 0 created


In [12]:
source.spectral_model("WCDA")

In [13]:
model_simu =  PowerLawSpectralModel(
    index=2.67,
    amplitude=3.4e-14 * u.Unit("cm-2 s-1 TeV-1"),
    reference=3* u.TeV,
)
model_source = SkyModel(spectral_model=model_simu, name="source")
analysis.get_spectrum_dataset(model_source)

In [14]:
print(analysis.spectrum_dataset)

SpectrumDataset
---------------

  Name                            : 0 

  Total counts                    : 9076 
  Total background counts         : 7679.46
  Total excess counts             : 1396.54

  Predicted counts                : 9053.02
  Predicted background counts     : 7679.46
  Predicted excess counts         : 1373.56

  Exposure min                    : 9.90e+07 m2 s
  Exposure max                    : 6.92e+11 m2 s

  Number of total bins            : 12 
  Number of fit bins              : 7 

  Fit statistic type              : cash
  Fit statistic value (-2 log(L)) : -130643.73

  Number of models                : 1 
  Number of parameters            : 3
  Number of free parameters       : 2

  Component 0: SkyModel
  
    Name                      : source
    Datasets names            : None
    Spectral model type       : PowerLawSpectralModel
    Spatial  model type       : 
    Temporal model type       : 
    Parameters:
      index                         : 

In [15]:
analysis.get_datasets()

In [16]:
print(analysis.datasets)

Datasets
--------

Dataset 0: 

  Type       : SpectrumDatasetOnOff
  Name       : obs-0
  Instrument : CTA
  Models     : 




In [17]:
analysis.datasets.info_table()

name,counts,excess,sqrt_ts,background,npred,npred_background,npred_signal,exposure_min,exposure_max,livetime,ontime,counts_rate,background_rate,excess_rate,n_bins,n_fit_bins,stat_type,stat_sum,counts_off,acceptance,acceptance_off,alpha
,,,,,,,,m2 s,m2 s,s,s,1 / s,1 / s,1 / s,,,,,,,,
str5,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64,int64,str5,float64,int64,float64,float64,float64
obs-0,9124,1411.1999999999998,14.18286665299302,7712.8,7948.000000000001,7948.000000000001,nan,99049939.36856084,692119568642.377,180000.0,180000.0,0.05068888888888889,0.04284888888888889,0.00784,12,7,wstat,211.65912933577988,38564,7.0,35.0,0.2


In [18]:
analysis.datasets.info_table()

name,counts,excess,sqrt_ts,background,npred,npred_background,npred_signal,exposure_min,exposure_max,livetime,ontime,counts_rate,background_rate,excess_rate,n_bins,n_fit_bins,stat_type,stat_sum,counts_off,acceptance,acceptance_off,alpha
,,,,,,,,m2 s,m2 s,s,s,1 / s,1 / s,1 / s,,,,,,,,
str5,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64,int64,str5,float64,int64,float64,float64,float64
obs-0,9124,1411.1999999999998,14.18286665299302,7712.8,7948.000000000001,7948.000000000001,nan,99049939.36856084,692119568642.377,180000.0,180000.0,0.05068888888888889,0.04284888888888889,0.00784,12,7,wstat,211.65912933577988,38564,7.0,35.0,0.2


In [19]:
analysis.set_models(Models(model_source))

INFO:feupy.analysis.core:Reading model.
INFO:feupy.analysis.core:Models

Component 0: SkyModel

  Name                      : source
  Datasets names            : None
  Spectral model type       : PowerLawSpectralModel
  Spatial  model type       : 
  Temporal model type       : 
  Parameters:
    index                         :      2.670   +/-    0.00             
    amplitude                     :   3.40e-14   +/- 0.0e+00 1 / (TeV s cm2)
    reference             (frozen):      3.000       TeV         




In [20]:
print(analysis.datasets)

Datasets
--------

Dataset 0: 

  Type       : SpectrumDatasetOnOff
  Name       : obs-0
  Instrument : CTA
  Models     : ['source']




In [21]:
analysis.get_file_name()

'sens_CTAO-South_40deg_50h_livetime50.0h'

In [22]:
analysis.run_fit()

In [23]:
analysis.fit_result.parameters.to_table()

type,name,value,unit,error,min,max,frozen,link,prior
str1,str9,float64,str14,float64,float64,float64,bool,str1,str1
,index,2.7562e+00,,1.040e-01,nan,nan,False,,
,amplitude,3.2483e-14,TeV-1 s-1 cm-2,3.247e-15,nan,nan,False,,
,reference,3.0000e+00,TeV,0.000e+00,nan,nan,True,,


In [24]:
analysis.get_flux_points()

In [25]:
analysis.flux_points

In [26]:
# Run the sensitivity analysis
analysis.compute_sensitivity()

# Write the sensitivity table to a file (CSV in this case)
analysis.write_table_sensitivity()

# Read the sensitivity table back if needed
sensitivity_table = analysis.read_table_sensitivity()

# Print the sensitivity table summary
print("\nSensitivity Table:")
print(sensitivity_table)

INFO:gammapy.estimators.points.core:Inferred format: gadf-sed
INFO:feupy.analysis.core:Table (sens_CTAO-South_40deg_50h_livetime50.0h.fits) stored to data.



Sensitivity Table:
      e_ref              e_min        ...     background      criterion  
       GeV                GeV         ...                                
------------------ ------------------ ... ------------------ ------------
 44.03397802866209 30.000000000000004 ...              740.6 significance
 94.86832980505132   64.6330407009565 ...            78819.6          bkg
204.38762071738844 139.24766500838334 ...  67938.40000000001          bkg
  440.339780286621  300.0000000000002 ...            16829.8          bkg
 948.6832980505134  646.3304070095651 ...  5421.200000000001          bkg
2043.8762071738831 1392.4766500838325 ... 1542.8000000000002          bkg
 4403.397802866211 3000.0000000000027 ...              405.0 significance
 9486.832980505136  6463.304070095652 ... 151.20000000000002 significance
20438.762071738816 13924.766500838328 ...  76.60000000000001 significance
 44033.97802866208 30000.000000000007 ...               36.0 significance
  94868.3298050513

In [27]:
table_sens = analysis.table_sens

In [ ]:
#from feupy.visualization.sensitivity import plot_sensitivity_from_table

plot_sensitivity_from_table(table_sens)

In [ ]:
table = analysis.table_sens
print(table[['e_ref', 'background', 'excess']])

In [ ]:
204388

In [ ]:
table = analysis.datasets.info_table()
table

In [ ]:
show_hist_counts(table)

In [ ]:
analysis.flux_points.data.to_table()

### 4. Inspect Results
Here, we review the results of the spectrum analysis, including the fit parameters and computed flux points.

In [ ]:
# Display the fit parameters
print("Fit Parameters:")
print(analysis.fit_result)

# Display the flux points
print("Flux Points:")
print(analysis.flux_points)

In [ ]:
analysis.flux_points.plot_fit()

## 5. Summary

In this notebook, we:

1. Selected a source from the 3HWC catalog.
2. Configured and ran a 1D ON/OFF spectrum analysis using `AnalysisSpectrum`.
3. Printed and saved the fit results and flux points for further analysis.

This modular design


## Sugestões de Exercício: Alterando a Configuração do `CTAOAnalysisConfig`

Explore diferentes aspectos da simulação espectral ajustando os parâmetros da configuração.  
Use as sugestões abaixo para gerar novos datasets e comparar os resultados com a configuração padrão.

---

### 1. Tempo de observação mais curto

```python
config.observation.livetime = 5 * u.h  # padrão era 50 h

# Exercício:
# Compare o espectro e os erros com tempos de 5 h, 10 h e 50 h.
# Como a estatística influencia o ajuste?

In [ ]:
# 2. Avaliar o modelo no centro da região ON

config.datasets.use_region_center = True  # padrão é False

# Exercício:
# Compare os espectros gerados com True e False.
# Isso afeta o fluxo normalizado ou o índice espectral?


In [ ]:
# 2. Avaliar o modelo no centro da região ON

config.datasets.use_region_center = True  # padrão é False

# Exercício:
# Compare os espectros gerados com True e False.
# Isso afeta o fluxo normalizado ou o índice espectral?


In [ ]:
# 3. Ativar empilhamento de observações 

config.statistics.n_obs = 20 

config.datasets.stack = True

# Exercício:
# Simule múltiplas observações de 10 h e compare o ajuste com e sem empilhamento.
# Qual abordagem recupera melhor os parâmetros?


In [ ]:
# 4. Alterar o método de máscara segura

config.datasets.safe_mask.methods = ["aeff-max"]
config.datasets.safe_mask.parameters = {"aeff_percent": 10}

# Exercício:
# Compare com o método "bkg-peak".
# A faixa de energia muda?
# O ajuste é mais confiável?
